# Laboratorio: De Parquet a Delta Lake

---

**Magister en Data Science - Universidad del Desarrollo**  
**Curso:** Big Data y Cloud Computing  
**Módulo 3:** Procesamiento Distribuido con Apache Spark  
**Sesión 3:** Viernes 29 de mayo de 2026
**Docente:** Luis Castillo Faune
**Alumno:** Ricardo Castro Vera
---

## Objetivo

Al finalizar este laboratorio serás capaz de:

- Medir empíricamente la diferencia entre transformaciones **Narrow** y **Wide** en Spark.
- Construir tu primer pipeline PySpark real: Bronze -> Silver con validacion de calidad.
- Identificar los **Shuffles** en un plan de ejecucion usando `.explain(True)`.
- Escribir datos en **Delta Lake** y consultar versiones anteriores con **time travel**.
- Conectar la experiencia con las decisiones de la **Fase 1 del proyecto**.

---

## Estructura

| Fase | Actividad |
|---|---|
| 1 | Setup, calentamiento y Narrow vs Wide |
| 2 | Pipeline Bronze->Silver con Delta Lake |
| 3 | Reflexion y conexion con el proyecto |
| 4 | Cierre |

---

## Dataset: NYC Yellow Taxi enero 2023
Fuente: [NYC Taxi & Limousine Commission](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page#)

| Característica | Valor |
|---|---|
| Filas aprox. | ~3.000.000 |
| Formato de entrada | Parquet |
| Formato de salida | Delta Lake (ACID + time travel) |

---
# FASE 1: Setup y calentamiento
## Sección 0: Instalación

Ejecuta esta celda primero. Tarda aproximadamente 2 minutos.

> Si aparece un error, espera que termine y **reinicia el runtime**: `Entorno de ejecución -> Reiniciar sesión`.

In [3]:
!pip install pyspark==3.5.1 delta-spark==3.2.0 -q
print('Dependencias instaladas correctamente')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
Dependencias instaladas correctamente


## Sección 1: SparkSession con soporte Delta Lake

Usaremos `configure_spark_with_delta_pip` para configurar Delta Lake.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
import time, os, urllib.request, warnings
warnings.filterwarnings('ignore')

builder = (
    SparkSession.builder
    .appName('Lab-BigData-BronzeToSilver')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'true')
    # Configuracion obligatoria para Delta Lake
    .config('spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

print(f'SparkSession inicializada')
print(f'  Version de Spark:  {spark.version}')
print(f'  Modo:              {spark.sparkContext.master}')
print(f'  Cores disponibles: {spark.sparkContext.defaultParallelism}')
print()
print('Delta Lake activo')

SparkSession inicializada
  Version de Spark:  3.5.1
  Modo:              local[*]
  Cores disponibles: 2

Delta Lake activo


## Sección 2: Descarga del dataset y capa Bronze

Descargamos el Parquet de NYC Taxi. Esta descarga simula la **ingesta** desde una fuente externa hacia la capa Bronze.

In [5]:
URL_DATASET = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet'
ARCHIVO_RAW = 'taxi_nyc_enero_2023.parquet'
RUTA_BRONZE = 'datos/bronze/taxi'
RUTA_SILVER = 'datos/silver/taxi_delta'

os.makedirs('datos/bronze', exist_ok=True)
os.makedirs('datos/silver', exist_ok=True)

if os.path.exists(ARCHIVO_RAW):
    print(f'Dataset ya descargado: {ARCHIVO_RAW}')
else:
    print('Descargando dataset NYC Taxi enero 2023...')
    t0 = time.time()
    urllib.request.urlretrieve(URL_DATASET, ARCHIVO_RAW)
    tam_mb = os.path.getsize(ARCHIVO_RAW) / 1e6
    print(f'Descarga completada en {time.time()-t0:.1f}s - {tam_mb:.1f} MB')

print()
print('Cargando en Spark como capa Bronze...')
t0 = time.time()

# Bronze: datos crudos con metadata de ingesta
df_bronze = (
    spark.read.parquet(ARCHIVO_RAW)
    .withColumn('_ingestion_ts', F.current_timestamp())
    .withColumn('_source_file',  F.lit(ARCHIVO_RAW))
)

n_bronze = df_bronze.count()
print(f'Capa Bronze: {n_bronze:,} filas en {time.time()-t0:.2f}s')
print(f'Columnas: {len(df_bronze.columns)}')
df_bronze.printSchema()

Descargando dataset NYC Taxi enero 2023...
Descarga completada en 0.3s - 47.7 MB

Cargando en Spark como capa Bronze...
Capa Bronze: 3,066,766 filas en 9.04s
Columnas: 21
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-

## Sección 3a: Calentamiento - primeras operaciones PySpark

In [6]:
# Vista previa
print('Primeros 5 registros de la capa Bronze:')
df_bronze.select('tpep_pickup_datetime','passenger_count','trip_distance','fare_amount','payment_type').show(5)

Primeros 5 registros de la capa Bronze:
+--------------------+---------------+-------------+-----------+------------+
|tpep_pickup_datetime|passenger_count|trip_distance|fare_amount|payment_type|
+--------------------+---------------+-------------+-----------+------------+
| 2023-01-01 00:32:10|            1.0|         0.97|        9.3|           2|
| 2023-01-01 00:55:08|            1.0|          1.1|        7.9|           1|
| 2023-01-01 00:25:04|            1.0|         2.51|       14.9|           1|
| 2023-01-01 00:03:48|            0.0|          1.9|       12.1|           1|
| 2023-01-01 00:10:29|            1.0|         1.43|       11.4|           1|
+--------------------+---------------+-------------+-----------+------------+
only showing top 5 rows



In [7]:
# Estadisticas descriptivas
print('Estadisticas descriptivas:')
df_bronze.select('passenger_count','trip_distance','fare_amount','tip_amount').describe().show()

Estadisticas descriptivas:
+-------+------------------+------------------+------------------+------------------+
|summary|   passenger_count|     trip_distance|       fare_amount|        tip_amount|
+-------+------------------+------------------+------------------+------------------+
|  count|           2995023|           3066766|           3066766|           3066766|
|   mean|1.3625321074328978|3.8473420306601414| 18.36706861234247|3.3679406710526827|
| stddev|0.8961199745510026|249.58375606858166|17.807821939337924| 3.826759457294151|
|    min|               0.0|               0.0|            -900.0|            -96.22|
|    max|               9.0|         258928.15|            1160.1|             380.8|
+-------+------------------+------------------+------------------+------------------+



In [8]:
# Distribucion de tipos de pago
print('Distribucion por tipo de pago en capa Bronze:')
df_bronze.groupBy('payment_type').count().orderBy('count', ascending=False).show()

n_invalidos = df_bronze.filter((F.col('fare_amount') <= 0) | (F.col('trip_distance') <= 0)).count()
print(f'Registros con fare_amount <= 0 o trip_distance <= 0: {n_invalidos:,} ({n_invalidos/n_bronze*100:.2f}%)')

Distribucion por tipo de pago en capa Bronze:
+------------+-------+
|payment_type|  count|
+------------+-------+
|           1|2411462|
|           2| 532241|
|           0|  71743|
|           4|  33297|
|           3|  18023|
+------------+-------+

Registros con fare_amount <= 0 o trip_distance <= 0: 68,384 (2.23%)


## Sección 3b: Narrow vs. Wide - medición empírica

Antes de construir el pipeline, medimos la diferencia de tiempo entre una transformación
**Narrow** (sin Shuffle) y una **Wide** (con Shuffle) sobre los mismos 3 millones de filas.

La clase teórica definio estos conceptos. Ahora los veremos aplicados a datos.

| Tipo | Qué hace Spark | Costo |
|---|---|---|
| **Narrow** | Cada partición se procesa localmente, sin mover datos entre nodos | Bajo |
| **Wide** | Las particiones intercambian datos entre Executors (Shuffle: red + disco) | Alto |

In [9]:
# Comparacion empirica: Narrow vs Wide
print('Comparacion: Narrow vs Wide sobre', f'{n_bronze:,}', 'filas')
print('=' * 52)

# NARROW: .filter()
# Cada particion se filtra localmente, sin ningun movimiento de datos
# entre Executors. Costo de red: cero.
print('NARROW: .filter(fare_amount > 0)')
print('  Cada particion se procesa localmente - sin Shuffle')
t0 = time.time()
n_filtrado = df_bronze.filter('fare_amount > 0').count()   # count() = accion
TIEMPO_NARROW = time.time() - t0
print(f'  Resultado: {n_filtrado:,} filas')
print(f'  Tiempo:    {TIEMPO_NARROW:.2f}s')
print()

# WIDE: .groupBy()
# Para agrupar por payment_type, todos los registros con el mismo valor
# deben estar en el mismo Executor. Spark los redistribuye -> SHUFFLE
print('WIDE: .groupBy(payment_type).count()')
print('  Spark redistribuye datos entre Executors -> SHUFFLE (red + disco)')
t0 = time.time()
r_wide = df_bronze.groupBy('payment_type').count().collect()  # collect() = accion
TIEMPO_WIDE = time.time() - t0
print(f'  Resultado: {len(r_wide)} grupos')
print(f'  Tiempo:    {TIEMPO_WIDE:.2f}s')
print()

factor = TIEMPO_WIDE / TIEMPO_NARROW if TIEMPO_NARROW > 0 else 0
print('=' * 52)
print(f'  Narrow (.filter):  {TIEMPO_NARROW:.2f}s')
print(f'  Wide (.groupBy):   {TIEMPO_WIDE:.2f}s')
print(f'  Factor:            {factor:.1f}x mas lento')
print('=' * 52)
print()
print('NOTA: En Colab (modo local) el impacto es menor porque no hay red real.')
print('Incluso una transformacion Wide podria ser mas rapida.')
print('En un cluster productivo con TBs de datos, este factor puede ser 100x.')

Comparacion: Narrow vs Wide sobre 3,066,766 filas
NARROW: .filter(fare_amount > 0)
  Cada particion se procesa localmente - sin Shuffle
  Resultado: 3,040,607 filas
  Tiempo:    1.00s

WIDE: .groupBy(payment_type).count()
  Spark redistribuye datos entre Executors -> SHUFFLE (red + disco)
  Resultado: 5 grupos
  Tiempo:    0.95s

  Narrow (.filter):  1.00s
  Wide (.groupBy):   0.95s
  Factor:            0.9x mas lento

NOTA: En Colab (modo local) el impacto es menor porque no hay red real.
Incluso una transformacion Wide podria ser mas rapida.
En un cluster productivo con TBs de datos, este factor puede ser 100x.


**Escribe aqui tu observación:**

> **El Wide fue ~0.9 veces más lento (es decir, prácticamente igual de rápido) porque:**
> Al ejecutar Spark en Google Colab (modo local [*]), todos los "executors" están en la misma máquina física. Esto significa que el "Shuffle" (intercambio de datos entre particiones para agrupar por payment_type) se realiza en la memoria o disco local sin tener que viajar por una red real. En un clúster de Big Data distribuido con múltiples nodos físicos, esta operación requeriría mover grandes volúmenes de datos a través de la red, lo que haría a la transformación Wide mucho más lenta que la Narrow (hasta 100x más lenta).

---
# FASE 2: Pipeline Bronze -> Silver

## Sección 4: Validación y reglas de calidad

In [10]:
print('Aplicando reglas de calidad de datos...')
print()

checks = [
    ('fare_amount > 0',         df_bronze.filter(F.col('fare_amount') <= 0).count()),
    ('trip_distance > 0',       df_bronze.filter(F.col('trip_distance') <= 0).count()),
    ('passenger_count entre 1-8', df_bronze.filter((F.col('passenger_count') <= 0) | (F.col('passenger_count') > 8)).count()),
    ('pickup < dropoff',        df_bronze.filter(F.col('tpep_pickup_datetime') >= F.col('tpep_dropoff_datetime')).count()),
]

for regla, n_desc in checks:
    print(f'  Regla {regla}: {n_desc:>8,} registros descartados')

n_validos = df_bronze.filter(
    (F.col('fare_amount') > 0) &
    (F.col('trip_distance') > 0) &
    (F.col('passenger_count').between(1, 8)) &
    (F.col('tpep_pickup_datetime') < F.col('tpep_dropoff_datetime'))
).count()

print()
print(f'  Total Bronze:    {n_bronze:>10,}')
print(f'  Estimado Silver: {n_validos:>10,} ({n_validos/n_bronze*100:.1f}% retencion)')

Aplicando reglas de calidad de datos...

  Regla fare_amount > 0:   26,159 registros descartados
  Regla trip_distance > 0:   45,862 registros descartados
  Regla passenger_count entre 1-8:   51,165 registros descartados
  Regla pickup < dropoff:    1,121 registros descartados

  Total Bronze:     3,066,766
  Estimado Silver:  2,884,165 (94.0% retencion)



El porcentaje de retención es **aceptable** porque un 94.0% indica que la gran mayoría de los datos crudos (capa Bronze) son de buena calidad y útiles para el negocio. Perder un 6% de registros debido a inconsistencias lógicas (como tarifas negativas, distancias cero o número de pasajeros inválido) es completamente normal y esperado en datasets del mundo real, y garantiza que la capa Silver contenga información confiable para el análisis y modelos de Machine Learning.

## Sección 5a: Transformaciones Silver (todas lazy)

Ninguna de estas lineas ejecuta nada: solo construyen el DAG. La ejecución ocurre en la Sección 6.

In [11]:
print('Construyendo plan Silver (evaluacion lazy)...')
t0 = time.time()

df_silver = (
    df_bronze
    # Filtros de calidad (NARROW)
    .filter(
        (F.col('fare_amount')   > 0) &
        (F.col('trip_distance') > 0) &
        (F.col('passenger_count').between(1, 8)) &
        (F.col('tpep_pickup_datetime') < F.col('tpep_dropoff_datetime'))
    )
    # Columnas de tiempo derivadas (NARROW)
    .withColumn('duracion_min',
        F.round((F.unix_timestamp(F.col('tpep_dropoff_datetime')) -
                 F.unix_timestamp(F.col('tpep_pickup_datetime'))) / 60, 1))
    .withColumn('hora_inicio',   F.hour('tpep_pickup_datetime'))
    .withColumn('dia_semana',    F.dayofweek('tpep_pickup_datetime'))
    .withColumn('es_fin_semana', F.col('dia_semana').isin([1, 7]))
    # Columnas de tarifa (NARROW)
    .withColumn('propina_pct',
        F.round(F.col('tip_amount') / F.col('fare_amount') * 100, 1))
    .withColumn('tarifa_por_milla',
        F.round(F.col('fare_amount') / F.col('trip_distance'), 2))
    # Filtro adicional (NARROW)
    .filter((F.col('duracion_min') > 0) & (F.col('duracion_min') < 180))
    # Introduciendo una transformacion WIDE explicita: repartition por tipo de pago
    #.repartition(F.col('payment_type')) # Esto forzara un Shuffle (Exchange)
    # Seleccion de columnas Silver (NARROW)
    .select(
        'tpep_pickup_datetime', 'tpep_dropoff_datetime',
        'passenger_count', 'trip_distance', 'payment_type',
        'fare_amount', 'tip_amount', 'total_amount',
        'duracion_min', 'hora_inicio', 'dia_semana', 'es_fin_semana',
        'propina_pct', 'tarifa_por_milla',
        '_ingestion_ts', '_source_file'
    )
)

print(f'Plan construido en {time.time()-t0:.4f}s')
print('(Ninguna transformacion se ejecuto todavia - evaluacion lazy)')
df_silver.printSchema()

Construyendo plan Silver (evaluacion lazy)...
Plan construido en 0.4066s
(Ninguna transformacion se ejecuto todavia - evaluacion lazy)
root
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- duracion_min: double (nullable = true)
 |-- hora_inicio: integer (nullable = true)
 |-- dia_semana: integer (nullable = true)
 |-- es_fin_semana: boolean (nullable = true)
 |-- propina_pct: double (nullable = true)
 |-- tarifa_por_milla: double (nullable = true)
 |-- _ingestion_ts: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)



## Sección 5b: Identificar Shuffles con .explain(True)

Antes de escribir, veamos el plan de ejecución. Cada nodo `Exchange` es un **Shuffle**.
Este plan muestra lo que vimos en el Bloque C de la parte teórica.

In [12]:
# Plan de ejecucion del DataFrame Silver
# Busca los nodos 'Exchange' en el output = cada uno es un Shuffle
print('Plan de ejecucion del DataFrame Silver:')
print('Cada nodo Exchange = un Shuffle (datos viajan entre Executors)')
print('=' * 60)
df_silver.explain(True)
print('=' * 60)
print()
print('Identifica en el plan:')
print('  NARROW - sin Exchange: Filter, Project, Scan')
print('         -> se ejecutan localmente por particion')
print('  WIDE   - con Exchange: si aparece en el plan')
print('         -> datos viajan por la red entre Executors')
print()
print('El pipeline Bronze->Silver tiene principalmente operaciones NARROW.')
print('El Shuffle controlado ocurre en la escritura particionada (.partitionBy).')

Plan de ejecucion del DataFrame Silver:
Cada nodo Exchange = un Shuffle (datos viajan entre Executors)
== Parsed Logical Plan ==
'Project ['tpep_pickup_datetime, 'tpep_dropoff_datetime, 'passenger_count, 'trip_distance, 'payment_type, 'fare_amount, 'tip_amount, 'total_amount, 'duracion_min, 'hora_inicio, 'dia_semana, 'es_fin_semana, 'propina_pct, 'tarifa_por_milla, '_ingestion_ts, '_source_file]
+- Filter ((duracion_min#720 > cast(0 as double)) AND (duracion_min#720 < cast(180 as double)))
   +- Project [VendorID#0L, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3, trip_distance#4, RatecodeID#5, store_and_fwd_flag#6, PULocationID#7L, DOLocationID#8L, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, airport_fee#18, _ingestion_ts#38, _source_file#60, duracion_min#720, hora_inicio#743, dia_semana#767, ... 3 more fields]
      +- Project [VendorID#0L, tpep_pickup_dat

## Sección 6: Escritura en Delta Lake

Esta es la primera **acción** del pipeline. Aquí Spark ejecuta el DAG completo.

> La escritura con `.partitionBy()` genera un Shuffle **controlado** y **necesario**: organiza los datos por `payment_type` para que las consultas futuras lean solo las particiones relevantes.

In [13]:
print('Escribiendo capa Silver en Delta Lake...')
print('(Esta celda dispara la ejecucion del DAG completo)')
print()

t0 = time.time()
(
    df_silver
    .write
    .format('delta')
    .mode('overwrite')
    .partitionBy('payment_type')
    .save(RUTA_SILVER)
)
TIEMPO_PIPELINE = time.time() - t0

# Verificar resultado
df_s = spark.read.format('delta').load(RUTA_SILVER)
n_silver = df_s.count()

print(f'Capa Silver escrita exitosamente en Delta Lake')
print(f'  Ruta:               {RUTA_SILVER}')
print(f'  Tiempo de pipeline: {TIEMPO_PIPELINE:.2f}s')
print()
print(f'  Filas Bronze:       {n_bronze:>10,}')
print(f'  Filas Silver:       {n_silver:>10,}')
print(f'  Descartados:        {n_bronze-n_silver:>10,} ({(n_bronze-n_silver)/n_bronze*100:.2f}%)')
print()

# Estructura en disco
print('Estructura de archivos en disco:')
for root, dirs, files in os.walk(RUTA_SILVER):
    level = root.replace(RUTA_SILVER, '').count(os.sep)
    indent = '  ' * level
    if level < 3:
        print(f'  {indent}{os.path.basename(root)}/')
        if level == 1:
            pq = [f for f in files if f.endswith('.parquet')]
            if pq: print(f'  {indent}  ({len(pq)} archivos Parquet)')

Escribiendo capa Silver en Delta Lake...
(Esta celda dispara la ejecucion del DAG completo)

Capa Silver escrita exitosamente en Delta Lake
  Ruta:               datos/silver/taxi_delta
  Tiempo de pipeline: 42.55s

  Filas Bronze:        3,066,766
  Filas Silver:        2,881,015
  Descartados:           185,751 (6.06%)

Estructura de archivos en disco:
  taxi_delta/
    payment_type=1/
      (1 archivos Parquet)
    payment_type=2/
      (1 archivos Parquet)
    _delta_log/
      _commits/
    payment_type=4/
      (1 archivos Parquet)
    payment_type=3/
      (1 archivos Parquet)


## Sección 7: Explorar la capa Silver

Ahora leeremos la capa Silver desde Delta Lake y haremos análisis sobre datos limpios.

In [14]:
print('Analisis sobre la capa Silver:')
print()

print('Tarifa y propina promedio por tipo de pago:')
(
    df_s.groupBy('payment_type')
    .agg(
        F.count('*').alias('viajes'),
        F.round(F.avg('fare_amount'), 2).alias('tarifa_prom'),
        F.round(F.avg('propina_pct'), 1).alias('propina_pct_prom'),
        F.round(F.avg('duracion_min'), 1).alias('duracion_prom_min'),
    )
    .orderBy('viajes', ascending=False)
).show()

print('Top 5 horas con mas viajes:')
df_s.groupBy('hora_inicio').count().orderBy('count', ascending=False).limit(5).show()

print('Propina promedio: fin de semana vs semana:')
df_s.groupBy('es_fin_semana').agg(
    F.count('*').alias('viajes'),
    F.round(F.avg('propina_pct'), 2).alias('propina_pct_prom')
).show()

Analisis sobre la capa Silver:

Tarifa y propina promedio por tipo de pago:
+------------+-------+-----------+----------------+-----------------+
|payment_type| viajes|tarifa_prom|propina_pct_prom|duracion_prom_min|
+------------+-------+-----------+----------------+-----------------+
|           1|2348517|      18.54|            26.0|             14.6|
|           2| 507003|      18.55|             0.0|             14.4|
|           4|  16299|      18.81|             0.1|             12.8|
|           3|   9196|      17.37|             0.0|             11.8|
+------------+-------+-----------+----------------+-----------------+

Top 5 horas con mas viajes:
+-----------+------+
|hora_inicio| count|
+-----------+------+
|         18|203402|
|         17|197029|
|         15|185232|
|         16|184452|
|         19|182508|
+-----------+------+

Propina promedio: fin de semana vs semana:
+-------------+-------+----------------+
|es_fin_semana| viajes|propina_pct_prom|
+-------------+-----

---
# FASE 3: Time Travel en Delta Lake

## Seccion 8: Historial de transacciones

Delta Lake mantiene un **transaction log** con el historial completo de escrituras. Como un `git log` para datos.

In [15]:
print('Historial de transacciones de la tabla Delta:')
DeltaTable.forPath(spark, RUTA_SILVER).history().select('version','timestamp','operation','operationMetrics').show(truncate=False)

Historial de transacciones de la tabla Delta:
+-------+-----------------------+---------+---------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                     |
+-------+-----------------------+---------+---------------------------------------------------------------------+
|0      |2026-06-02 23:29:10.867|WRITE    |{numFiles -> 4, numOutputRows -> 2881015, numOutputBytes -> 60676400}|
+-------+-----------------------+---------+---------------------------------------------------------------------+



In [16]:
# Simular segunda carga (append)
print('Simulando segunda carga de datos (append de 50.000 registros)...')
(
    df_silver.limit(50000)
    .write.format('delta').mode('append')
    .partitionBy('payment_type')
    .save(RUTA_SILVER)
)
print('Segunda carga completada')
print()
print('Historial actualizado:')
DeltaTable.forPath(spark, RUTA_SILVER).history().select('version','timestamp','operation').show()

Simulando segunda carga de datos (append de 50.000 registros)...
Segunda carga completada

Historial actualizado:
+-------+--------------------+---------+
|version|           timestamp|operation|
+-------+--------------------+---------+
|      1|2026-06-02 23:30:...|    WRITE|
|      0|2026-06-02 23:29:...|    WRITE|
+-------+--------------------+---------+



In [17]:
# Time travel
print('Time Travel en accion:')
print()

df_v0 = spark.read.format('delta').option('versionAsOf', 0).load(RUTA_SILVER)
df_v1 = spark.read.format('delta').option('versionAsOf', 1).load(RUTA_SILVER)
df_actual = spark.read.format('delta').load(RUTA_SILVER)

n_v0, n_v1, n_actual = df_v0.count(), df_v1.count(), df_actual.count()

print(f'  Version 0 (carga inicial):    {n_v0:>10,} filas')
print(f'  Version 1 (despues del append): {n_v1:>8,} filas')
print(f'  Version actual:               {n_actual:>10,} filas')
print()

avg_v0 = df_v0.agg(F.round(F.avg('fare_amount'),2)).collect()[0][0]
avg_v1 = df_v1.agg(F.round(F.avg('fare_amount'),2)).collect()[0][0]
print(f'  Tarifa promedio v0: ${avg_v0}')
print(f'  Tarifa promedio v1: ${avg_v1}')
print()
print('El time travel permite:')
print('  -> Auditar cambios en los datos')
print('  -> Reentrenar modelos ML con snapshots historicos')
print('  -> Recuperarse de una escritura incorrecta (rollback)')

Time Travel en accion:

  Version 0 (carga inicial):     2,881,015 filas
  Version 1 (despues del append): 2,931,015 filas
  Version actual:                2,931,015 filas

  Tarifa promedio v0: $18.54
  Tarifa promedio v1: $18.58

El time travel permite:
  -> Auditar cambios en los datos
  -> Reentrenar modelos ML con snapshots historicos
  -> Recuperarse de una escritura incorrecta (rollback)


In [18]:
# Spark SQL sobre Delta
df_v0.createOrReplaceTempView('silver_v0')
print('Consulta SQL directa sobre la capa Silver (version 0):')
spark.sql("""
    SELECT
        CASE payment_type
            WHEN 1 THEN 'Tarjeta'
            WHEN 2 THEN 'Efectivo'
            WHEN 3 THEN 'Sin cargo'
            WHEN 4 THEN 'Disputa'
            ELSE 'Otro'
        END AS tipo_pago,
        COUNT(*) AS viajes,
        ROUND(AVG(fare_amount), 2) AS tarifa_prom,
        ROUND(AVG(propina_pct), 1) AS propina_pct_prom,
        ROUND(AVG(duracion_min), 1) AS duracion_min
    FROM silver_v0
    GROUP BY payment_type
    ORDER BY viajes DESC
""").show()

Consulta SQL directa sobre la capa Silver (version 0):
+---------+-------+-----------+----------------+------------+
|tipo_pago| viajes|tarifa_prom|propina_pct_prom|duracion_min|
+---------+-------+-----------+----------------+------------+
|  Tarjeta|2348517|      18.54|            26.0|        14.6|
| Efectivo| 507003|      18.55|             0.0|        14.4|
|  Disputa|  16299|      18.81|             0.1|        12.8|
|Sin cargo|   9196|      17.37|             0.0|        11.8|
+---------+-------+-----------+----------------+------------+



---
# FASE 4: Exploración libre y reflexión

## Sección 9: Escribe tu propio código

**Pregunta 9.1** - ¿Qué día de la semana genera más ingresos totales?

Pista: usa `dia_semana` y `F.sum('total_amount')`.

In [19]:
ingresos_por_dia = (
    df_s
    .groupBy('dia_semana')
    .agg(F.sum('total_amount').alias('ingresos_totales'))
    .orderBy('ingresos_totales', ascending=False)
)
ingresos_por_dia.show()

+----------+--------------------+
|dia_semana|    ingresos_totales|
+----------+--------------------+
|         1|1.3047829540000424E7|
|         3|1.2698904910000643E7|
|         5|1.1339579490000129E7|
|         6|1.1138089000000298E7|
|         2|1.0906445260000287E7|
|         7|1.0611407920000337E7|
|         4|1.0543515240000185E7|
+----------+--------------------+



**Respuesta 9.1:**

De acuerdo con la ejecución del código, el día de la semana que genera mayores ingresos totales es el **día 1**, que en el estándar de Spark corresponde al **Domingo** aprox. $13,047,829.

**Pregunta 9.2** - ¿A qué hora del dia se registran las tarifas más altas por milla?

Pista: usa `hora_inicio` y `F.avg('tarifa_por_milla')`.

In [21]:
tarifa_por_hora = (
    df_s
    .groupBy('hora_inicio')
    .agg(F.avg('tarifa_por_milla').alias('tarifa_promedio_por_milla'))
    .orderBy('tarifa_promedio_por_milla', ascending=False)
)
tarifa_por_hora.show(24)

+-----------+-------------------------+
|hora_inicio|tarifa_promedio_por_milla|
+-----------+-------------------------+
|          4|       15.340507056395401|
|          5|        14.78395389655981|
|         16|       11.245353580925913|
|         15|       11.017077584794682|
|         14|       10.898177242864673|
|          3|       10.869626151728093|
|         13|       10.711506572647506|
|         12|       10.518254676610862|
|         18|       10.323787081739392|
|          9|       10.281076596876852|
|         19|       10.221916617810528|
|          8|       10.192130555453632|
|         10|       10.176648285382283|
|         17|       10.159916739679984|
|         11|       10.044078828373024|
|         23|        9.872430404774974|
|          0|        9.767257197127138|
|         21|        9.730681127076814|
|          1|        9.694142372881462|
|          6|        9.633845491874055|
|          7|        9.619864068252054|
|         22|        9.276063116032239|


## Sección 10: Reflexión final

### Pregunta 10.1 - Calidad de datos

In [22]:
print('=' * 55)
print('   RESUMEN DEL PIPELINE BRONZE -> SILVER')
print('=' * 55)
print(f'   Registros Bronze:    {n_bronze:>12,}')
print(f'   Registros Silver:    {n_silver:>12,}')
print(f'   Descartados:         {n_bronze-n_silver:>12,}')
print(f'   Retencion:           {n_silver/n_bronze*100:>10.1f}%')
print(f'   Tiempo de pipeline:  {TIEMPO_PIPELINE:>10.2f}s')
print(f'   Tiempo Narrow:       {TIEMPO_NARROW:>10.2f}s')
print(f'   Tiempo Wide:         {TIEMPO_WIDE:>10.2f}s')
print(f'   Factor Wide/Narrow:  {TIEMPO_WIDE/TIEMPO_NARROW:>10.1f}x')
print('=' * 55)

   RESUMEN DEL PIPELINE BRONZE -> SILVER
   Registros Bronze:       3,066,766
   Registros Silver:       2,881,015
   Descartados:              185,751
   Retencion:                 93.9%
   Tiempo de pipeline:       42.55s
   Tiempo Narrow:             1.00s
   Tiempo Wide:               0.95s
   Factor Wide/Narrow:         0.9x


**Escribe aquí tu respuesta:**

> **% de retención:** 93.9%
>
> **¿Es aceptable?:** Sí, es completamente aceptable. En datasets reales (como los viajes de taxi ingresados manualmente o por sensores), una tasa de error cercana al 6% por valores atípicos, nulos o ilógicos (viajes de 0 km, tarifas negativas) es normal. Filtrar estos datos asegura que la capa Silver tenga información de calidad.
>
> **¿Qué harías si el descarte supera el 20%?:** Si el descarte es tan alto, indicaría un problema sistémico. Primero, detendría el pipeline y analizaría los registros descartados para encontrar patrones (¿hay un sensor fallando? ¿un cambio en el sistema de origen?). Luego, levantaría una alerta al equipo responsable de la fuente de datos (data producers) y revisaría si nuestras reglas de calidad son demasiado estrictas o si realmente hubo una degradación masiva en la captura de datos.

### Pregunta 10.2 - Conexión con tu proyecto

**a)** *¿Cuántos Shuffles tiene el pipeline Bronze->Silver que construiste hoy? ¿En que transformación ocurren? ¿Eran necesarios o podrías haberlos evitado?*

**b)** *Vuelve a leer tu Fase 1. ¿Cambiarías alguna decisión de formato, ingesta o arquitectura de capas después de haber ejecutado este pipeline?*

**Escribe aquí tu respuesta:**

> **a) Shuffles identificados:** El pipeline Bronze->Silver tiene **1 solo Shuffle** (representado como un nodo Exchange en el plan físico). Este ocurre al final, durante la acción de escritura, específicamente por la instrucción ".partitionBy('payment_type')". Sí, era **necesario y justificado**, ya que particionar los datos en disco optimiza enormemente las futuras lecturas (Partition Pruning), permitiendo que consultas futuras por tipo de pago ignoren los archivos irrelevantes. Podría haberse evitado guardando sin particionar, pero penalizaría el rendimiento de la capa Silver.
>
> **b) Decisión que cambiaría:** Definitivamente cambiaría el formato de almacenamiento de los datos procesados. En la Fase 1 probablemente consideré CSV o JSON, pero tras ver Delta Lake, usaría este formato para las capas Silver y Gold por sus beneficios: esquema estricto, compresión en columnas, transacciones ACID y el soporte nativo para *Time Travel* (esencial para auditorías o reproducibilidad en Machine Learning).
>
> **Lo que mantendría igual:** Mantendría la arquitectura de capas (Medallion Architecture). Conservar una capa Bronze inmutable y cruda demostró ser fundamental, ya que nos permitió aplicar filtros de calidad y crear la capa Silver sin miedo a perder los datos originales en caso de un error de lógica.

---
## Cierre

In [23]:
spark.stop()
print('SparkSession cerrada correctamente.')
print()
print('Guarda una copia en tu Google Drive: Archivo -> Guardar una copia en Drive')
print()
print('En la proxima sesion (5 de junio):')
print('  Modulo 4 - Servicios Gestionados de Nube para Big Data')
print('  Conectaremos este pipeline a GCP.')
print('  Será necesario crear tu cuenta GCP.')
print()
print('Pregunta abierta para la Clase 5 (Optimizacion y FinOps):')
print('  Hoy viste el Shuffle en .explain(). En la Clase 5 veremos:')
print('  -> Como diagnosticarlo en Spark UI')
print('  -> Como reducirlo con particionamiento y caching')
print('  -> Cuanto cuesta en dinero en Dataproc')

SparkSession cerrada correctamente.

Guarda una copia en tu Google Drive: Archivo -> Guardar una copia en Drive

En la proxima sesion (5 de junio):
  Modulo 4 - Servicios Gestionados de Nube para Big Data
  Conectaremos este pipeline a GCP.
  Será necesario crear tu cuenta GCP.

Pregunta abierta para la Clase 5 (Optimizacion y FinOps):
  Hoy viste el Shuffle en .explain(). En la Clase 5 veremos:
  -> Como diagnosticarlo en Spark UI
  -> Como reducirlo con particionamiento y caching
  -> Cuanto cuesta en dinero en Dataproc


---

## Referencias

- Damji et al. (2020). *Learning Spark* (2nd ed., caps. 2-4). O'Reilly.
- Delta Lake Documentation. https://docs.delta.io/latest/
- Apache Spark Documentation. https://spark.apache.org/docs/latest/sql-programming-guide.html
- NYC TLC Trip Record Data. https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

---

*Magister en Data Science - Universidad del Desarrollo - Big Data y Cloud Computing - 2026*